# Aula 14 — Laboratório: vazamento e reprodutibilidade

**Consultoria Especializada em Ciência de Dados 1**

Até aqui você mediu, validou, comparou e calibrou. Hoje você faz a pergunta que vem antes de
tudo isso: **o número que você mediu é verdadeiro?** Um vazamento não dá erro, não quebra o
código. Ele entrega um resultado bom.

Duas bases, as duas **simuladas** para esta aula:

- `alunos_semestre.csv` — 1000 alunos de uma disciplina. A coordenação quer saber, **na semana 6**,
  quem vai reprovar, para oferecer monitoria enquanto ainda dá tempo. É o exemplo dos slides.
- `vendas_diarias.csv` — as vendas de uma loja, dia a dia, em 2024 e 2025. Ela entra na Parte 4,
  porque vazamento temporal só existe quando a base tem data.

Em cada parte há um **exemplo resolvido** e, logo depois, um **exercício**. Nos exercícios com
**Explique**, escreva com as suas palavras na célula de resposta: é ali que se vê se a ideia ficou.

## Parte 0 — Ambiente e dados

O protocolo é o das aulas anteriores: 80% para treino e validação, 20% guardados para o teste,
validação cruzada com 10 dobras estratificadas. A função `pipe(colunas)` monta o Pipeline:
one-hot nas colunas de texto, padronização nas numéricas e regressão logística no fim. Você vai
chamá-la várias vezes, mudando só a lista de colunas.

In [5]:
%pip install numpy pandas joblib scikit-learn matplotlib


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

## Parte 1 — A pergunta, coluna por coluna

Antes de qualquer modelo, a auditoria. Para cada coluna, uma pergunta só:
**eu teria esta coluna, deste jeito, no momento da previsão?** Aqui o momento é a **semana 6**.

### Exemplo — duas colunas, a mesma pergunta

A nota da P1 sai na semana 5: na semana 6 ela já existe, **fica**. A nota da P2 só sai na semana
12: na semana 6 ela ainda não existe, **sai**. Repare que a decisão não depende de quão boa a
coluna é para prever, só de **quando ela nasce**.

In [10]:
quando_nasce = {"nota_P1": "semana 5", "nota_P2": "semana 12"}
for col, quando in quando_nasce.items():
    decisao = "fica" if col == "nota_P1" else "sai"
    print(f"{col:10s} nasce na {quando:9s} -> {decisao}")

nota_P1    nasce na semana 5  -> fica
nota_P2    nasce na semana 12 -> sai


### Exercício 1 — a auditoria das dez colunas

Complete o dicionário `decisao` com `"fica"`, `"sai"` ou `"perguntar"` para cada coluna de `X`.
A informação de quando cada coluna nasce está na tabela abaixo. A célula monta sozinha a lista
`HONESTAS`, que você vai usar no resto do laboratório.

| coluna | quando nasce |
|---|---|
| `curso`, `semestre_ingresso` | na matrícula |
| `nota_P1` | semana 5 |
| `faltas_ate_semana_6` | semana 6 |
| `frequentou_monitoria` | não se sabe: pode ter sido antes ou depois da semana 6 |
| `nota_P2` | semana 12 |
| `faltas_no_semestre` | fim do semestre |
| `fez_substitutiva` | fim do semestre, e só para quem foi mal |
| `media_final` | fim do semestre |
| `sala_prova_final` | fim do semestre (as salas são sorteadas) |

Pela regra da aula, uma coluna marcada como `"perguntar"` **não entra** enquanto ninguém
confirmar quando ela nasce: na dúvida, remova.

**Explique:** escolha a coluna que foi mais difícil de decidir e escreva por que ela gerou
dúvida e o que você faria para resolver.

In [14]:
# pylint: disable=wrong-import-position,import-error

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (
    KFold,
    TimeSeriesSplit,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


ModuleNotFoundError: No module named 'joblib'

**Resposta:**

## Parte 2 — O impacto, em número

Agora você mede o que a auditoria evitou.

### Exemplo — o modelo com tudo o que está no arquivo

É o que acontece quando alguém tira só a coluna alvo e treina com o resto. Nada dá erro.

In [ ]:
TODAS = [c for c in X.columns if c != "frequentou_monitoria"]
print(f"acurácia (CV) com todas as colunas: {acuracia_cv(TODAS):.3f}")

### Exercício 2 — o número honesto e a linha de base

Calcule a acurácia na validação cruzada **só com as colunas `HONESTAS`**. Depois calcule a
acurácia de quem chuta sempre a classe mais comum, com `DummyClassifier(strategy="most_frequent")`
nas mesmas dobras.

Responda: qual é o número honesto? Quanto ele fica abaixo do modelo com todas as colunas, e
quanto fica acima do chute? Por que o número do exemplo não serve para a coordenação na semana 6?

In [ ]:
acc_honesta = ____

base = DummyClassifier(strategy="most_frequent")
acc_chute = cross_val_score(____, X_tr[HONESTAS], y_tr, cv=cv, scoring="accuracy").mean()

print(f"todas as colunas : {acuracia_cv(TODAS):.3f}")
print(f"só as honestas   : {acc_honesta:.3f}")
print(f"chute            : {acc_chute:.3f}")

**Resposta:**

## Parte 3 — Vazar não é o mesmo que inflar

Uma coluna pode vazar e ajudar muito, ou vazar e não ajudar nada. A regra é a mesma para as duas.

### Exemplo — uma coluna do fim do semestre, sozinha

Junte a nota da P2 às colunas honestas e meça. Calcule também a correlação dela com o alvo.

In [ ]:
corr = np.corrcoef(X_tr["nota_P2"], y_tr)[0, 1]
print(f"correlação nota_P2 x reprovou: {corr:+.3f}")
print(f"honestas           : {acuracia_cv(HONESTAS):.3f}")
print(f"honestas + nota_P2 : {acuracia_cv(HONESTAS + ['nota_P2']):.3f}")

### Exercício 3 — as cinco colunas do fim, uma de cada vez

Repita o exemplo para **cada coluna de `FIM`**: junte só ela às honestas e meça a acurácia.
Calcule a correlação com o alvo quando a coluna for numérica (para `sala_prova_final`, que é
texto, deixe `None`). Monte uma tabela.

**Explique:** a sala da prova final quase não muda a acurácia. Por que ela sai mesmo assim? E
por que as faltas do semestre também quase não mexem no número, se faltar muito reprova?

In [ ]:
linhas = []
for col in FIM:
    acc = ____
    corr = None if col in TEXTO else ____
    linhas.append({"coluna acrescentada": col, "acurácia (CV)": round(acc, 3),
                   "correlação com o alvo": None if corr is None else round(corr, 3)})

tabela = pd.DataFrame(linhas)
print(f"referência, só as honestas: {acuracia_cv(HONESTAS):.3f}")
tabela

**Resposta:**

## Parte 4 — Vazamento temporal

Aqui a base muda, de propósito: na base da escola todas as linhas são do mesmo semestre, não há
passado e futuro entre um aluno e outro. Na base de vendas há. O objetivo é prever as vendas de
um dia, e o erro é medido em **MAE**: quantas vendas, em média, a previsão erra.

In [ ]:
vendas = pd.read_csv("datasets/vendas_diarias.csv", parse_dates=["data"])
vendas["dia_semana"] = vendas["data"].dt.dayofweek      # 0 = segunda, 6 = domingo
vendas["dia"] = np.arange(len(vendas))                  # 0, 1, 2, ... na ordem do calendário

plt.figure(figsize=(9, 3))
plt.plot(vendas["data"], vendas["vendas"], color="#0F766E", lw=0.8)
plt.ylabel("vendas no dia"); plt.grid(alpha=.25); plt.tight_layout(); plt.show()
print(len(vendas), "dias, de", vendas["data"].min().date(), "a", vendas["data"].max().date())

### Exemplo — embaralhar os dias × validar para frente

A mesma floresta, as mesmas colunas, duas formas de dividir. O `KFold` embaralhado põe dias de
março no teste e deixa agosto e dezembro no treino. O `TimeSeriesSplit` treina sempre no passado
e testa no que vem logo depois.

In [ ]:
floresta = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=0)
cols = ["dia_semana", "dia"]

def mae(modelo, dados, colunas, divisao):
    s = cross_val_score(modelo, dados[colunas], dados["vendas"], cv=divisao,
                        scoring="neg_mean_absolute_error")
    return -s.mean()

embaralhado = KFold(n_splits=5, shuffle=True, random_state=42)
para_frente = TimeSeriesSplit(n_splits=5)
print(f"KFold embaralhado : erra {mae(floresta, vendas, cols, embaralhado):5.2f} vendas por dia")
print(f"TimeSeriesSplit   : erra {mae(floresta, vendas, cols, para_frente):5.2f} vendas por dia")

### Exercício 4 — a média do mês e a média até ontem

Crie duas colunas:

- `media_mes`: a média de vendas **do mês inteiro** em que o dia está. Use
  `groupby(vendas["data"].dt.to_period("M"))["vendas"].transform("mean")`.
- `media_30d`: a média dos **30 dias anteriores**, sem contar o próprio dia. Use
  `vendas["vendas"].shift(1).rolling(30).mean()`.

Tire as linhas em que `media_30d` ficou vazia (os 30 primeiros dias) e compare as duas com uma
`LinearRegression`, sempre com `TimeSeriesSplit` e com `dia_semana` junto.

**Explique:** qual das duas colunas parece melhor? Por que ela não pode ser usada para prever as
vendas de amanhã? Use o dia 3 de janeiro como exemplo.

In [ ]:
vendas["media_mes"] = ____
vendas["media_30d"] = ____

v = vendas.dropna().reset_index(drop=True)
linear = LinearRegression()
print(f"com media_30d (até ontem)    : erra {mae(linear, v, ['dia_semana', ____], para_frente):5.2f}")
print(f"com media_mes (mês inteiro)  : erra {mae(linear, v, ['dia_semana', ____], para_frente):5.2f}")

**Resposta:**

## Parte 5 — A semente: o mesmo experimento, o mesmo número

Volta a base da escola, já com as colunas honestas.

### Exemplo — três divisões sem semente

Rode a célula duas ou três vezes e veja os números mudarem. Nada no modelo mudou: só o sorteio de
quem cai no teste.

In [ ]:
for rodada in range(3):
    Xa, Xb, ya, yb = train_test_split(X[HONESTAS], y, test_size=0.2, stratify=y)   # sem random_state
    print(f"rodada {rodada + 1}: acurácia no teste {pipe(HONESTAS).fit(Xa, ya).score(Xb, yb):.3f}")

### Exercício 5 — vinte sementes

Repita a divisão com `random_state` de 0 a 19 e guarde a acurácia no teste de cada uma. Mostre a
menor, a maior, a diferença entre elas e a média.

**Explique:** um colega rodou as 20, viu que a semente 18 dá a maior acurácia e quer entregar o
modelo com ela. Por que isso é escolher olhando o teste? E por que a sua resposta deve vir com
uma semente fixa, qualquer que seja?

In [ ]:
acc_por_semente = []
for semente in range(20):
    Xa, Xb, ya, yb = train_test_split(X[HONESTAS], y, test_size=0.2,
                                      random_state=____, stratify=y)
    acc_por_semente.append(____)

acc_por_semente = np.array(acc_por_semente)
print(f"menor : {acc_por_semente.min():.3f} (semente {acc_por_semente.argmin()})")
print(f"maior : {acc_por_semente.max():.3f} (semente {acc_por_semente.argmax()})")
print(f"diferença: {acc_por_semente.max() - acc_por_semente.min():.3f}")
print(f"média : {acc_por_semente.mean():.3f}")

**Resposta:**

## Parte 6 — Salvar o modelo inteiro

### Exemplo — o que o `dump` grava e o que o `load` devolve

- `joblib.dump(objeto, arquivo)` grava o objeto inteiro num arquivo. Aqui o objeto é o Pipeline
  já treinado: o one-hot, o padronizador e a regressão, cada um com o que aprendeu no treino.
- `joblib.load(arquivo)` faz o caminho de volta: lê o arquivo e devolve o mesmo objeto, pronto
  para prever, sem treinar de novo.

Para provar que o modelo veio mesmo do arquivo, a célula **apaga a variável** antes de carregar.
Depois de carregar, confira sempre uma previsão conhecida.

In [ ]:
modelo = pipe(HONESTAS).fit(X_tr[HONESTAS], y_tr)
previsoes_antes = modelo.predict(X_te[HONESTAS])

# 1. dump: grava o Pipeline treinado num arquivo
joblib.dump(modelo, "modelo_reprovacao.joblib")
print("arquivo criado:", os.path.getsize("modelo_reprovacao.joblib"), "bytes")

# 2. apaga o modelo da memória: daqui para frente ele só existe no arquivo
del modelo

# 3. load: lê o arquivo e devolve o Pipeline pronto
carregado = joblib.load("modelo_reprovacao.joblib")
print("o que voltou do arquivo:", list(carregado.named_steps))
print("médias que o padronizador aprendeu no treino (nota_P1, faltas):",
      carregado.named_steps["pre"].named_transformers_["num"].mean_.round(2))

# 4. confira: as mesmas previsões de antes, sem treinar de novo
print("mesmas previsões de antes?", (carregado.predict(X_te[HONESTAS]) == previsoes_antes).all())
print(f"acurácia no teste: {carregado.score(X_te[HONESTAS], y_te):.3f}")

### Exercício 6 — o erro de salvar só o classificador

Na semana 6 chegam 20 alunos novos (use os 20 primeiros do teste). Alguém guardou só a regressão,
`carregado.named_steps["modelo"]`, e refez o pré-processamento **à mão**, ajustando um one-hot e um
padronizador novos **nesses 20 alunos**.

Compare a probabilidade de reprovar dada pelo Pipeline salvo com a dada pelo caminho à mão.
Mostre a maior diferença e quantos alunos trocaram de classe (corte em 0,5).

**Explique:** por que nenhum erro apareceu? E por que o mesmo aluno recebeu outra probabilidade?

In [ ]:
lote = X_te[HONESTAS].head(20)
so_classificador = carregado.named_steps["modelo"]

pre_a_mao = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["curso", "semestre_ingresso"]),
    ("num", StandardScaler(), ["nota_P1", "faltas_ate_semana_6"])])
pre_a_mao.fit(____)                      # o erro: ajustar o pré-processamento nos dados novos

p_certo = carregado.predict_proba(lote)[:, 1]
p_mao   = so_classificador.predict_proba(pre_a_mao.transform(lote))[:, 1]

comparacao = pd.DataFrame({"nota_P1": lote["nota_P1"].values,
                           "P(reprovar) Pipeline": p_certo.round(3),
                           "P(reprovar) à mão": p_mao.round(3)})
print(f"maior diferença: {np.abs(p_certo - p_mao).max():.3f}")
print(f"trocaram de classe: {____}")
comparacao

**Resposta:**

## Parte 7 — A model card (a ficha do modelo)

**O que é.** Um texto curto que acompanha o modelo entregue e responde às perguntas que qualquer
pessoa vai fazer antes de usá-lo: para que ele serve, com que dados foi feito, quanto ele acerta,
o que foi tirado e por quê, e quando ele **não** deve ser usado.

**A analogia.** É a bula do modelo. A bula de um remédio diz para que ele serve, a dose, os
efeitos e a validade. Ninguém toma um remédio só porque o comprimido está na caixa; ninguém
deveria usar um modelo só porque o `.joblib` está na pasta.

**Por que existe.** O arquivo `.joblib` da Parte 6 guarda o modelo, mas não guarda as decisões.
Daqui a seis meses, quem abrir o arquivo — um colega, a coordenação ou você mesmo — não vai lembrar
por que a nota da P2 ficou de fora nem de onde veio o número de desempenho. A ficha guarda isso.

**Onde fica.** Num arquivo de texto, `README.md`, na mesma pasta do `.joblib`. Não precisa de
programa nenhum.

| campo | o que responde |
|---|---|
| **Objetivo** | para que o modelo serve e **em que momento** ele é usado |
| **Dados** | de onde vieram, quantas linhas, **quais colunas entram** |
| **Desempenho** | a métrica, **onde** foi medida (validação, teste) e a comparação com o chute |
| **Removidas** | que colunas saíram e **por quê** — é a auditoria da Parte 1 por escrito |
| **Limitações** | quando o modelo **não** deve ser usado |
| **Versão** | data, semente e nome do arquivo, para alguém conseguir refazer |

### Exemplo — a ficha de um modelo de vendas

A ficha do modelo linear da Parte 4, que prevê as vendas do dia com `dia_semana` e `media_30d`:

> **Objetivo:** prever as vendas de cada dia da loja, usando só o que se sabe até a véspera.
> **Dados:** `vendas_diarias.csv`, 2024 e 2025, uma linha por dia. Colunas: `dia_semana` e
> `media_30d` (média dos 30 dias anteriores).
> **Desempenho:** erra em média 13,72 vendas por dia (MAE), validando para frente com
> `TimeSeriesSplit` de 5 dobras: o teste está sempre depois do treino.
> **Removidas:** `media_mes`, porque usa dias do mês que ainda não aconteceram. Com ela o erro
> parecia menor, mas era vazamento.
> **Limitações:** não testado em datas especiais nem fora de 2024–2025.
> **Versão:** v1 · 10/11/2026 · modelo linear, sem sorteio · arquivo `modelo_vendas.joblib`.

### Exercício 7 — a ficha do modelo de reprovação

Escreva, na célula de resposta, a model card do modelo que você salvou na Parte 6. Use os
mesmos campos do exemplo e os **seus números**: a acurácia na validação (Exercício 2) e no teste
(Parte 6), as colunas que saíram e por quê, e o que ficou de fora por dúvida.

**Explique**, no fim da ficha, em uma ou duas frases: para quem essa ficha serve daqui a seis
meses?

**Resposta:**

---

**Parabéns!** Você auditou as colunas antes de treinar, mediu quanto um vazamento infla a
métrica, separou vazar de inflar, viu o futuro entrar no treino pelo k-fold embaralhado e por
uma média mal calculada, mediu o quanto a semente mexe no número, salvou o modelo inteiro e
escreveu a ficha dele.

Para levar ao projeto: **audite antes de treinar, na dúvida remova, e documente o que saiu.**

Na próxima aula (17/11): **seleção de atributos** — escolher, de propósito, quais colunas entram.